## this contains the implementation of various excercises questions for video 1

### common imports & global values

In [1]:
from dataclasses import dataclass,field
import numpy as np
import torch
import torch.nn.functional as F
from abc import ABC, abstractmethod

In [2]:
LEARNING_RATE = 50
REGULARIZATION_VALUE = 0.01
SMOOTHING_COUNT = 3
BOUNDARY_CHAR = "."
SEED_VAL = 2147483647

In [3]:
@dataclass
class Vocab:
    """Vocab is a packaging class which contains important information for language modelling tasks

        Members:
            boundary_char: str - this signifies the start and end of a sequence
            vocab_letters: list[str] - this signifies the list of unique characters that occur in the data set
            stoi: dict[str, int] - this is the mapping between the the unique characters and int values
            itos: dict[int, str] -  this is the reverse of stoi
            n_unique: int - this is the number of unique characters + the boundary character
    """
    boundary_char: str = ""
    vocab_letters: list[str] = field(default_factory=list)
    stoi: dict[str, int] = field(default_factory=dict)
    itos: dict[int, str] = field(default_factory=dict)
    n_unique: int = field(default=0)


In [4]:
@dataclass
class NGram:
    data: list[tuple[str, ...]] = field(default_factory=list)
    n: int = field(default=2)

In [5]:
def get_names()->list[str]:
    """Read the names file and load the contentents into memory as a list"""
    with open('names.txt', 'r') as file:
        return file.read().splitlines()

In [6]:
def create_vocab(boundary_char: str, data_set: list[str])->Vocab:
  """Creates a vocab object for the provided boundary character and list of strings


    Args:
      boundar_char: str - the start and end of a sequence
      data_set: list[str] - list of character sequences that are to be modelled
    
    Returns
      vocab: Vocab - this is the vocab object that contains important character level model information.
  """


  # create a list of unique occuring characters from the set
  unique_chars = sorted(list(set(''.join(data_set))))
  
  # add boundary char at index 0
  unique_chars.insert(0, boundary_char)



  # create the string to int mapping for the unqiue characters
  stoi = {
    s: i
    for i, s in enumerate(unique_chars[1:], start = 1)
  }

  # give the special boundary character the int value of 0
  stoi[boundary_char] = 0

  # reverse the above mapping to have int -> string
  itos = {
    i : s
    for s, i in stoi.items()
  }


  # package all together as a object to be used in other parts of the program
  return Vocab(
    boundary_char= boundary_char,
    vocab_letters= unique_chars[1:],
    stoi = stoi,
    itos=itos,
    n_unique=len(unique_chars)
  )
  



In [7]:
def generate_n_grams(data_set: list[str], boundary_char: str, n: int = 2)->NGram:
    """Function to generate the n-gram set of the input names dataset by default it generates bigram examples

    Args:
        data_set: list[str] = this is a list of strings (names)
        boundar_char : str = this is appended at the start and the end of the string
        n: int = if 2 we generate bigrams, 3 we generate tri-grams etc.

    Returns:
        n_grams = list[tupe[str,...]] -> this is a list of tuple each tuple will atleast have two elements (the input character and the resultant character) in case of bigrams
    """

    # resultant list
    n_grams = []

    for name in data_set:
        
        # append the boundary char
        char_list = [boundary_char] + list(name) + [boundary_char]

        for i in range(len(char_list) - n + 1):
            n_grams.append(tuple(char_list[i : i + n]))
    return NGram(
        n = n,
        data=n_grams
    )


In [48]:
class NGramInterface(ABC):
    def __init__(self, data_set: list[str], boundary_char: str, n: int, seed_val)->None:
        self.data_set = data_set
        self.vocab = create_vocab(data_set=data_set, boundary_char=boundary_char)
        self.n_gram = generate_n_grams(data_set= data_set, boundary_char=boundary_char, n = n)
        self.counts = self.create_counts_array()
        self.probs = (self.counts + 1).float()
        self.probs /= self.probs.sum(dim=-1, keepdim=True)
        self.generator = torch.Generator().manual_seed(seed_val)
    
    @abstractmethod
    def create_counts_array(self)->torch.Tensor:
        raise NotImplementedError(f"Implement for inferencing the counting method")
    
    @abstractmethod
    def sample_names_using_counting_method(self, sample_size: int = 5)->None:
        raise NotImplementedError("Implement this to sample using counting method")
    
    @abstractmethod
    def calculate_nll_counting_method(self)->None:
        raise NotImplementedError("Implement this for calculating loss for the counting method")
    

In [82]:
class Bigram(NGramInterface):
    def __init__(self, data_set: list[str], boundary_char: str, seed_val: int)->None:
        super().__init__(data_set=data_set, boundary_char=boundary_char, n = 2, seed_val=seed_val)
    
    def create_counts_array(self)->torch.Tensor:
        counts = torch.zeros((self.vocab.n_unique,) * self.n_gram.n, dtype=torch.int32)
        for ch1, ch2 in self.n_gram.data:
            ix1, ix2 = self.vocab.stoi[ch1], self.vocab.stoi[ch2]
            counts[ix1, ix2] += 1
        return counts
    def sample_names_using_counting_method(self, sample_size:int = 5):
        for _ in range(sample_size):
            ix = 0
            out = []

            while True:
                p = self.probs[ix]
                ix = torch.multinomial(p, num_samples=1, replacement=True, generator=self.generator).item()
                out.append(self.vocab.itos[ix])
                if ix == 0:
                    break
            print(''.join(out))
    def calculate_nll_counting_method(self)->None:
        counts = 0
        log_likelihood = 0
        for ch1, ch2 in self.n_gram.data:
            ix1, ix2 = self.vocab.stoi[ch1], self.vocab.stoi[ch2]
            p = self.probs[ix1, ix2]
            log_prob = p.log()
            counts += 1
            log_likelihood += log_prob.item()
        neg_log_likelihood = -log_likelihood
        neg_log_likelihood /= counts
        print(f'{neg_log_likelihood=:.4f}')




In [90]:
class Trigram(NGramInterface):
    def __init__(self, data_set: list[str], boundary_char: str, seed_val: int):
        super().__init__(data_set=data_set, boundary_char=boundary_char, n = 3, seed_val=seed_val)
    def create_counts_array(self)->torch.Tensor:
        counts =  torch.zeros(size=(self.vocab.n_unique,) * self.n_gram.n, dtype=torch.int32)
        for ch1, ch2, ch3 in self.n_gram.data:
            ix1, ix2, ix3 = self.vocab.stoi[ch1], self.vocab.stoi[ch2], self.vocab.stoi[ch3]
            counts[ix1, ix2, ix3] += 1
        return counts
    def sample_names_using_counting_method(self, sample_size = 5):
        for _ in range(sample_size):
            ix1, ix2 = 0, 0
            out = [self.vocab.itos[ix1]]
            while True:
                p = self.probs[ix1, ix2]
                ix3 = torch.multinomial(p, replacement=True, num_samples=1, generator=self.generator).item()
                out.append(self.vocab.itos[ix3])
                if ix3 == 0:
                    break
                ix1, ix2 = ix2, ix3
            print(''.join(out))
    def calculate_nll_counting_method(self)->None:
        counts = 0
        log_likelihood = 0
        for ch1, ch2, ch3 in self.n_gram.data:
            ix1, ix2, ix3 = self.vocab.stoi[ch1], self.vocab.stoi[ch2], self.vocab.stoi[ch3]
            p = self.probs[ix1, ix2, ix3]
            log_prob = p.log()
            counts += 1
            log_likelihood += log_prob.item()
        neg_log_likelihood = -log_likelihood
        neg_log_likelihood /= counts
        print(f'{neg_log_likelihood=:.4f}')
        

        

In [91]:
# get the names
names = get_names()

In [92]:
bigrams = Bigram(data_set=names, boundary_char=BOUNDARY_CHAR, seed_val=2147483647)
trigrams = Trigram(data_set=names, boundary_char=BOUNDARY_CHAR, seed_val=2147483647)

In [93]:
bigrams.probs[0].sum(), trigrams.probs[0][0].sum()

(tensor(1.), tensor(1.))

In [94]:
bigrams.sample_names_using_counting_method()

cexze.
momasurailezitynn.
konimittain.
llayn.
ka.


In [95]:
trigrams.sample_names_using_counting_method()

.ce.
.za.
.zogh.
.uriana.
.kaydnevonimittain.


In [96]:
bigrams.calculate_nll_counting_method()

nll=2.4546


In [97]:
trigrams.calculate_nll_counting_method()

neg_log_likelihood=2.0931
